<a href="https://colab.research.google.com/github/prince9939367489/Diffusion-Model-Based-Cancelable-Biometric-Templates/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# INSTALL LIBRARIES
# =========================
!pip install tensorflow scikit-learn opencv-python xgboost

# =========================
# IMPORT LIBRARIES
# =========================
import os
import numpy as np
import cv2
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# =========================
# MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# DATASET PATH
# =========================
data_path = "/content/drive/MyDrive/Eye dataset"

# =========================
# LOAD DATA
# =========================
IMG_SIZE = 224

def load_data(data_dir):
    X, y = [], []
    classes = os.listdir(data_dir)

    for label, class_name in enumerate(classes):
        class_path = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            try:
                img = cv2.imread(os.path.join(class_path, img_name))
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                X.append(img)
                y.append(label)
            except:
                continue

    return np.array(X), np.array(y)

X, y = load_data(data_path)

# Normalize labels
y = y - np.min(y)

print("Total Images:", len(X))
print("Total Classes:", len(set(y)))

# =========================
# PREPROCESS
# =========================
X = preprocess_input(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# FEATURE EXTRACTION
# =========================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)

feature_model = Model(inputs=base_model.input, outputs=x)

base_model.trainable = True

print("Extracting Features...")
train_features = feature_model.predict(X_train, batch_size=32)
test_features = feature_model.predict(X_test, batch_size=32)

# =========================
# CANCELABLE BIOMETRIC METHODS
# =========================
def iom_hash(features, k=5):
    np.random.seed(42)
    R = np.random.randn(features.shape[1], 256)
    projected = np.dot(features, R)
    return np.argsort(projected, axis=1)[:, -k:]

def biohashing(features):
    np.random.seed(42)
    R = np.random.randn(features.shape[1], features.shape[1])
    bio = np.dot(features, R)
    return (bio > 0).astype(int)

def random_projection(features):
    np.random.seed(42)
    R = np.random.randn(features.shape[1], 128)
    return np.dot(features, R)

cb_train = {
    "IoM": iom_hash(train_features),
    "BioHash": biohashing(train_features),
    "RP": random_projection(train_features)
}

cb_test = {
    "IoM": iom_hash(test_features),
    "BioHash": biohashing(test_features),
    "RP": random_projection(test_features)
}

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate_model(name, model, X_tr, X_te):
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{name}:")
    if hasattr(model, "best_params_"):
        print("Best Params:", model.best_params_)
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)

# =========================
# MAIN LOOP
# =========================
for cb_name in cb_train:
    print(f"\n===== CB Technique: {cb_name} =====")

    X_tr = cb_train[cb_name].reshape(cb_train[cb_name].shape[0], -1)
    X_te = cb_test[cb_name].reshape(cb_test[cb_name].shape[0], -1)

    #  RandomForest
    rf = GridSearchCV(
        RandomForestClassifier(),
        {
            'n_estimators': [100, 200],
            'max_depth': [None, 10],
        },
        cv=3, n_jobs=-1
    )
    evaluate_model("RandomForest (Tuned)", rf, X_tr, X_te)

    #  SVM (BEST)
    svm = GridSearchCV(
        SVC(),
        {
            'C': [1, 10],
            'gamma': ['scale', 0.01],
            'kernel': ['rbf']
        },
        cv=3, n_jobs=-1
    )
    evaluate_model("SVM (Tuned)", svm, X_tr, X_te)

    #  KNN
    knn = GridSearchCV(
        KNeighborsClassifier(),
        {
            'n_neighbors': [3, 5],
        },
        cv=3, n_jobs=-1
    )
    evaluate_model("KNN (Tuned)", knn, X_tr, X_te)

    #  XGBoost (POWERFUL)
    xgb = RandomizedSearchCV(
        XGBClassifier(
            objective='multi:softmax',
            num_class=len(set(y_train)),
            eval_metric='mlogloss',
            use_label_encoder=False
        ),
        {
            'n_estimators': [100, 200],
            'max_depth': [3, 5],
            'learning_rate': [0.01, 0.1]
        },
        n_iter=5,
        cv=3,
        n_jobs=-1
    )
    evaluate_model("XGBoost (Tuned)", xgb, X_tr, X_te)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Images: 14161
Total Classes: 4
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Extracting Features...
354/354 ━━━━━━━━━━━━━━━━━━━━ 899s 3s/step
89/89 ━━━━━━━━━━━━━━━━━━━━ 237s 3s/step

===== CB Technique: IoM =====

RandomForest (Tuned):
Best Params: {'max_depth': None, 'n_estimators': 100}
Accuracy : 0.7511471937875044
Precision: 0.7507202422633473
Recall   : 0.7511471937875044
F1 Score : 0.7502661318261603

SVM (Tuned):
Best Params: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Accuracy : 0.6392516766678433
Precision: 0.6691036348101985
Recall   : 0.6392516766678433
F1 Score : 0.6264191048001166

KNN (Tuned):
Best Params: {'n_neighbors': 3}
Accuracy : 0.616307800917755
Precision: 0.6192660199484686
Recall   : 0.616307800917755
F1 Score : 0.6142149060990642


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:24:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (Tuned):
Best Params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Accuracy : 0.7020825979527003
Precision: 0.7036266919493671
Recall   : 0.7020825979527003
F1 Score : 0.7014511795418881

===== CB Technique: BioHash =====

RandomForest (Tuned):
Best Params: {'max_depth': None, 'n_estimators': 200}
Accuracy : 0.9459936463113308
Precision: 0.9457517572345606
Recall   : 0.9459936463113308
F1 Score : 0.9455173153129854

SVM (Tuned):
Best Params: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Accuracy : 0.9527003176844334
Precision: 0.9525813849439089
Recall   : 0.9527003176844334
F1 Score : 0.9525856358746135

KNN (Tuned):
Best Params: {'n_neighbors': 3}
Accuracy : 0.9346981997882103
Precision: 0.9343288285605307
Recall   : 0.9346981997882103
F1 Score : 0.9343838859409022


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:32:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (Tuned):
Best Params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Accuracy : 0.9315213554535827
Precision: 0.9310842696516738
Recall   : 0.9315213554535827
F1 Score : 0.930950770678508

===== CB Technique: RP =====

RandomForest (Tuned):
Best Params: {'max_depth': None, 'n_estimators': 200}
Accuracy : 0.9431697846805507
Precision: 0.9427890263803806
Recall   : 0.9431697846805507
F1 Score : 0.9428087379286247

SVM (Tuned):
Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Accuracy : 0.9491704906459584
Precision: 0.9489841666523495
Recall   : 0.9491704906459584
F1 Score : 0.9489978200120638

KNN (Tuned):
Best Params: {'n_neighbors': 3}
Accuracy : 0.939286974938228
Precision: 0.9388421334872453
Recall   : 0.939286974938228
F1 Score : 0.9389696391938848


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:40:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (Tuned):
Best Params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Accuracy : 0.935757147899753
Precision: 0.9354923894253936
Recall   : 0.935757147899753
F1 Score : 0.9354133299756803


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
